# AI Architectural Milestones

This notebook breaks down the mathematical evolution of modern AI. From the earliest perceptron to modern Transformers, almost all architectures rely on chained **Linear Transformations** (i.e. $y = Wx + b$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display

class LinearTransform:
    def __init__(self, w, b):
        self.w = np.array(w, dtype=float)
        self.b = np.array(b, dtype=float)

    def forward(self, x):
        return np.dot(self.w, x) + self.b

### Mathematical Visualization (SymPy)

We can natively render the algebraic matrices making up the formula $Wx + b$ using SymPy's visual rendering.

In [ ]:
w_sym = sp.Matrix([[1.5, 0.5], [-0.5, 1.0]])
x_sym = sp.Matrix([2.0, 1.0])
b_sym = sp.Matrix([1.0, -1.0])

print("W matrix:")
display(w_sym)
print("x input:")
display(x_sym)
print("b bias:")
display(b_sym)
print("W * x + b Result:")
display(w_sym * x_sym + b_sym)

### Geometric Intuition (Matplotlib)

**What are linear transformations good for?**
They are mathematical operations that map input vectors to new spaces. By stretching, rotating, and shifting data points, a linear transformation (like $y = Wx + b$) can project complex data into a space where it is easily understandable. Good examples include rotating an image in computer graphics, compressing features via PCA dimensionality reduction, or moving abstract word representations closer together in NLP.

**What is happening?**
The input data point is being physically moved, rotated, and stretched by the network's matrix layer.

**What are we visualizing?**
The blue arrow is our starting `x_input` vector. The red arrow shows the new transformed output vector after it has been acted upon by weight matrix $W$ and shifted by bias $b$.

In [ ]:
class VectorVisualizer:
    def __init__(self, w_matrix=[[1.5, 0.5], [-0.5, 1.0]], b_vector=[1.0, -1.0], x_input=[2.0, 1.0], title="Example 1"):
        self.w_matrix = w_matrix
        self.b_vector = b_vector
        self.x_input = x_input
        self.title = title

    def run(self):
        w = np.array(self.w_matrix)
        b = np.array(self.b_vector)
        x = np.array(self.x_input)
        y = np.dot(w, x) + b
        
        plt.figure(figsize=(6, 6))
        plt.axhline(0, color='gray', linestyle='--')
        plt.axvline(0, color='gray', linestyle='--')
        plt.quiver(0, 0, x[0], x[1], angles='xy', scale_units='xy', scale=1, color='blue', label='Input x')
        plt.quiver(0, 0, y[0], y[1], angles='xy', scale_units='xy', scale=1, color='red', label='Transformed Yield (Wx+b)')
        
        max_limit = max(abs(np.concatenate([x, y]))) + 2
        plt.xlim(-max_limit, max_limit)
        plt.ylim(-max_limit, max_limit)
        plt.grid()
        plt.legend()
        plt.title(f'Geometric Intuition: {self.title}')
        plt.show()

# Example 1: Expanding Vector
visualizer_1 = VectorVisualizer(w_matrix=[[1.5, 0.5], [-0.5, 1.0]], b_vector=[1.0, -1.0], x_input=[2.0, 1.0], title="Example 1: Expanding Vector")
visualizer_1.run()

# Example 2: Shrinking & Shifting Vector
visualizer_2 = VectorVisualizer(w_matrix=[[0.5, -0.2], [0.1, 0.5]], b_vector=[0.0, 2.0], x_input=[2.0, 1.0], title="Example 2: Shrinking & Shifting Vector")
visualizer_2.run()

## 1. Perceptron (1958)

The **Perceptron** represents the genesis of artificial neural networks. Mathematically, it operates precisely like our standard Linear Transformation: it computes a dot product of inputs, scales them by weights, adds a bias, and passes the sum into a discrete step function (often $1$ if $z > 0$ else $0$). 

**Visual Check:** A single perceptron essentially attempts to draw a straight line (a 'decision boundary') perfectly splitting two classes of data in space.

In [ ]:
class PerceptronDemo(LinearTransform):
    def __init__(self, w=[-1.0, 1.0], b=[0.5], x=None):
        # We define a 2D line: -1.0*x1 + 1.0*x2 + 0.5 = 0
        super().__init__(w=w, b=b)
        self.x = x

    def run(self):
        np.random.seed(42)
        X = self.x if self.x is not None else np.random.randn(50, 2) * 2
        
        colors = []
        for x in X:
            z = self.forward(x)
            output = 1 if z > 0 else 0
            colors.append('red' if output == 1 else 'blue')
            
        plt.figure(figsize=(6,5))
        plt.scatter(X[:,0], X[:,1], c=colors, edgecolor='k')
        
        x1_vals = np.array([-5, 5])
        x2_vals = (-self.w[0]*x1_vals - self.b[0]) / self.w[1]
        plt.plot(x1_vals, x2_vals, 'k--', label='Decision Boundary ($Wx + b = 0$)')
        
        plt.fill_between(x1_vals, x2_vals, 6, color='red', alpha=0.1)
        plt.fill_between(x1_vals, -6, x2_vals, color='blue', alpha=0.1)
        
        plt.xlim(-5, 5)
        plt.ylim(-5, 5)
        plt.legend()
        plt.title('Perceptron Visual Decision Boundary')
        plt.show()

PerceptronDemo().run()

## 2. Backpropagation (1986)

Hidden layers solved the Perceptron's non-linear limits, but training those layers mathematically was practically impossible. **Backpropagation** algorithmically calculates gradients backward utilizing the Chain Rule of calculus. It systematically tells the model how far 'off' each weight mathematically is from perfect accuracy.

**Visual Check:** Below we plot Loss over 20 iterations. Because gradients point strictly strictly 'downhill', you can visibly watch the optimization rapidly conquer the mean-squared-error penalty.

In [ ]:
class BackpropagationDemo(LinearTransform):
    def __init__(self):
        super().__init__(w=[[10.0, 5.0]], b=[50.0])
        # x = [Rooms, Age (years)]
        self.x = np.array([3.0, 10.0])
        self.y_true = np.array([300.0]) # Target House Price
        self.lr = 0.001

    def run(self):
        losses = []
        epochs = 20
        for _ in range(epochs):
            y_pred = self.forward(self.x)
            loss = 0.5 * np.sum((y_pred - self.y_true)**2)
            losses.append(loss)
            
            grad_y = y_pred - self.y_true
            grad_w = np.outer(grad_y, self.x)
            grad_b = grad_y
            
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            
        plt.figure(figsize=(6,4))
        plt.plot(range(1, epochs+1), losses, marker='o', linestyle='-', color='purple')
        plt.title('Backpropagation Gradient Descent Optimization')
        plt.xlabel('Epochs')
        plt.ylabel('Mean Squared Error Loss')
        plt.grid(True)
        plt.show()

BackpropagationDemo().run()

## 3. CNNs / AlexNet (2012)

Instead of plugging every single pixel into a dense line of transformations, **Convolutional Neural Networks (CNNs)** scan patches dynamically using much smaller, shared 'Filters'. This captures specific spatial structures (edges, corners) globally and proved that Deep Learning scaling via GPUs could crush traditional image tasks (see: AlexNet, 2012).

**Visual Check:** We display the discrete pixel matrix being 'convolved' against a tiny 2x2 edge-detector Filter.

In [ ]:
class CNNDemo:
    def __init__(self):
        from sklearn.datasets import load_digits
        digits = load_digits()
        self.image = digits.images[0] / 16.0
        self.filter = np.array([
            [1, -1],
            [1, -1]
        ])

    def run(self):
        h, w = self.image.shape
        fh, fw = self.filter.shape
        output = np.zeros((h - fh + 1, w - fw + 1))
        for i in range(output.shape[0]):
            for j in range(output.shape[1]):
                patch = self.image[i:i+fh, j:j+fw]
                output[i, j] = np.sum(patch * self.filter)
                
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(self.image, cmap='Blues')
        axes[0].set_title('Original 8x8 Image')
        axes[1].imshow(self.filter, cmap='Oranges')
        axes[1].set_title('2x2 Edge Filter')
        axes[2].imshow(output, cmap='Greens')
        axes[2].set_title('7x7 Convoluted Feature Map')
        plt.tight_layout()
        plt.show()

CNNDemo().run()

## 4. RNNs / LSTMs (2014-2016)

To handle sequenced arrays (like spoken sentences), **Recurrent Neural Networks (RNNs)** were designed with literal memory banks. A hidden representation matrix calculates its new output dynamically based on BOTH the current token and what was processed dynamically in the previous tick of time.

**Visual Check:** Below we plot an unrolled time-series trajectory tracking the activations of two internal 'hidden nodes' as they ingest sequence vectors iteratively.

In [ ]:
class RNNDemo:
    def __init__(self):
        self.transform_x = LinearTransform(w=[[0.5, -0.2], [0.1, 0.8]], b=[0.0, 0.0])
        self.transform_h = LinearTransform(w=[[0.7, 0.1], [-0.1, 0.6]], b=[0.1, -0.1])
        self.sequence = [np.array([1.0, 0.0]) if i%2==0 else np.array([0.0, 1.0]) for i in range(15)]

    def run(self):
        hidden_states = []
        hidden = np.array([0.0, 0.0])
        for x_t in self.sequence:
            z = self.transform_x.forward(x_t) + self.transform_h.forward(hidden)
            hidden = np.tanh(z)
            hidden_states.append(hidden)
            
        hidden_states = np.array(hidden_states)
        plt.figure(figsize=(7,4))
        plt.plot(hidden_states[:, 0], label='Hidden Node 1', marker='s')
        plt.plot(hidden_states[:, 1], label='Hidden Node 2', marker='^')
        plt.title('RNN Hidden State Dynamics Over Time')
        plt.xlabel('Time Sequence Tick')
        plt.ylabel('Internal Activation Intensity')
        plt.grid(True)
        plt.legend()
        plt.show()

RNNDemo().run()

## 5. Transformers (2017)

**Transformers** proved RNN-style memory was a sequential gridlock bottleneck! Their breakthrough lies in mapping sequence tokens indiscriminately in parallel via 'Attention'. They evaluate every piece of an input against every other piece, calculating a massive relational proximity score grid dynamically against Query, Key, and Value components.

**Visual Check:** Below is an Attention Matrix Heatmap. Notice visually how darker cells indicate the exact tokens providing mathematical 'attention' to other contextual tokens across the sequence.

In [ ]:
class TransformerDemo:
    def __init__(self):
        self.x = np.array([
            [1.0, 0.0, 0.1], 
            [0.1, 0.9, 0.1], 
            [0.2, 0.1, 0.8], 
            [0.9, 0.2, 0.0]  
        ])
        self.q_proj = LinearTransform(w=np.eye(3) * 0.5, b=[0]*3)
        self.k_proj = LinearTransform(w=np.eye(3) * 0.5, b=[0]*3)

    def run(self):
        Q = np.array([self.q_proj.forward(row) for row in self.x])
        K = np.array([self.k_proj.forward(row) for row in self.x])
        
        scores = np.dot(Q, K.T) / np.sqrt(3)
        exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
        attn_w = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        
        words = ["The", "cat", "sat", "down"]
        
        plt.figure(figsize=(6,5))
        plt.imshow(attn_w, cmap='magma')
        for i in range(len(words)):
            for j in range(len(words)):
                plt.text(j, i, f'{attn_w[i, j]:.2f}', ha="center", va="center", color="black" if attn_w[i,j] > 0.5 else "white")
        plt.colorbar(label='Self-Attention Relational Weight')
        plt.xticks(range(len(words)), words)
        plt.yticks(range(len(words)), words)
        plt.title('Transformer Attention Weights Heatmap')
        plt.ylabel('Token Querying Context')
        plt.xlabel('Token Being Attended To')
        plt.show()

TransformerDemo().run()